In [5]:
"""
---------------------------------------------------------
Practical: Distributed Evolutionary Algorithm using DEAP

Problem Statement:
Implement DEAP (Distributed Evolutionary Algorithms) using Python.
The program should simulate two populations evolving separately
and exchanging individuals to demonstrate distributed evolution.
---------------------------------------------------------
"""

import random
from deap import base, creator, tools

# --------------------------------
# Step 1: Define Fitness and Individual
# --------------------------------
# Avoid error if running multiple times (Jupyter safety)
if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))

if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Each individual = binary list (0 or 1)
toolbox.register("attr_bool", random.randint, 0, 1)

# Create individual of size 10
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, 10)

# Create population (list of individuals)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# --------------------------------
# Step 2: Define Fitness Function
# --------------------------------
# Objective: Maximize number of 1s
def fitness(individual):
    return (sum(individual),)

toolbox.register("evaluate", fitness)

# Mutation: flip bits randomly
toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)

# --------------------------------
# Step 3: Create TWO populations
# (Distributed Concept)
# --------------------------------
pop1 = toolbox.population(n=5)
pop2 = toolbox.population(n=5)

print("Initial Population 1:", pop1)
print("Initial Population 2:", pop2)

# --------------------------------
# Step 4: Evolution Process
# --------------------------------
for gen in range(5):
    print("\nGeneration:", gen)

    # Evaluate fitness of all individuals
    for ind in pop1 + pop2:
        ind.fitness.values = toolbox.evaluate(ind)

    # Selection: choose best individuals
    pop1 = sorted(pop1, key=lambda x: x.fitness.values[0], reverse=True)
    pop2 = sorted(pop2, key=lambda x: x.fitness.values[0], reverse=True)

    best1 = pop1[:2]   # top 2 from population 1
    best2 = pop2[:2]   # top 2 from population 2

    # Crossover: create children from best individuals
    child1 = creator.Individual(best1[0])
    child2 = creator.Individual(best2[0])

    # Mutation: introduce small random changes
    if random.random() < 0.2:
        toolbox.mutate(child1)
    if random.random() < 0.2:
        toolbox.mutate(child2)

    # Evaluate new children
    child1.fitness.values = toolbox.evaluate(child1)
    child2.fitness.values = toolbox.evaluate(child2)

    # Distributed Exchange:
    # Swap individuals between populations
    pop1 = best1 + [child1, child2]
    pop2 = best2 + [child2, child1]

    print("Population 1:", pop1)
    print("Population 2:", pop2)

# --------------------------------
# Step 5: Find Best Solution
# --------------------------------
all_pop = pop1 + pop2

best = max(all_pop, key=lambda x: x.fitness.values[0])

print("\nBest Individual:", best)
print("Fitness Value:", best.fitness.values[0])

Initial Population 1: [[0, 1, 1, 1, 0, 1, 1, 1, 0, 0], [0, 1, 0, 1, 0, 1, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 0, 1], [1, 0, 1, 0, 1, 1, 0, 1, 1, 1], [1, 1, 0, 1, 0, 1, 1, 1, 0, 0]]
Initial Population 2: [[1, 0, 0, 1, 1, 0, 1, 1, 0, 1], [1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [0, 1, 0, 1, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 1, 1, 1, 1, 0], [1, 0, 1, 0, 1, 1, 1, 0, 1, 0]]

Generation: 0
Population 1: [[0, 1, 0, 1, 0, 1, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 0, 1], [0, 1, 1, 1, 0, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 0, 1, 1, 0, 0]]
Population 2: [[1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [1, 0, 0, 1, 1, 0, 1, 1, 0, 1], [1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [0, 1, 1, 1, 0, 1, 1, 1, 1, 1]]

Generation: 1
Population 1: [[0, 1, 1, 1, 0, 1, 1, 1, 1, 1], [0, 1, 0, 1, 0, 1, 1, 1, 1, 1], [0, 1, 1, 1, 0, 1, 1, 1, 1, 1], [1, 0, 1, 1, 0, 1, 1, 0, 1, 1]]
Population 2: [[0, 1, 1, 1, 0, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [1, 0, 1, 1, 0, 1, 1, 0, 1, 1], [0, 1, 1, 1, 0, 1, 1, 1, 1, 1]]

Generation: 2
Population 1: [[0